This Source Code Form is subject to the terms of the Mozilla Public  License, v. 2.0. If a copy of the MPL was not distributed with this file, You can obtain one at http://mozilla.org/MPL/2.0/.


In [ ]:
import os
import pathlib
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import tomlkit
import torch
from IPython.display import Image, clear_output
from IPython.display import display as ipy_display
from torch.utils.data import DataLoader
from tqdm import tqdm

import s4casting.factories as fc
from s4casting.core.cli import get_configuration
from s4casting.core.config import Configuration
from s4casting.core.context import Context
from s4casting.core.distributions import gmm_to_quantiles

In [ ]:
# helpful functions
def count_parameters(model: torch.nn.Module):
    """Returns total and trainable number of parameters in a model.

    Args:
        model: torch.nn.Module

    Returns:
        total: total number of parameters
        trainable: number of trainable parameters
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def mb(n_params: int, dtype_bytes: int = 4):
    """Converts number of parameters to MB given dtype size.

    Args:
        n_params: number of parameters
        dtype_bytes: size of data type in bytes (default: 4 for float32)

    Returns:
        memory in MB
    """
    return n_params * dtype_bytes / (1024**2)


# plotting color function
def get_quantile_alpha(quantile_value, median=0.5):
    """Returns opacity based on distance from median.

    Closer to median = darker (higher alpha), further from median = lighter (lower alpha).
    Maps distance [0, 0.5] to alpha [0.6, 0.1].

    Args:
        quantile_value: quantile value (between 0 and 1)
        median: median quantile value (default: 0.5)

    Returns:
        alpha: opacity value (between 0.1 and 0.6)
    """
    distance = abs(quantile_value - median)
    return 0.6 - (distance / 0.5) * 0.45

## s4casting with historical load measurements

Hello and welcome to a simple tutorial introducing s4casting - the other notebooks introduce the coding aspects of the repository while this one provides further details on the model itself.

In this notebook, we'll fall back to simple loops and matplotlib to show the training and predictions made by a small model. 

We'll end by showing how the model improves in performance with scale - larger model and more data. 

> **Note#1**
> If your kernel gets disconnected and the cells 'hang' like this -> [*], click on 'Kernel' > 'Reconnect to Kernel'

> **Note#2**
> Please make sure to download the data by running ```bash data/get_data.sh``` before starting on this tutorial]

## Build model

We'll start by building the model and taking a good look at it - what does the input/output look like, what are the components of the model. What is configurable? 

In [ ]:
# Build model - for a detailed description of these functions, please refer to the 01_training notebook
# You can safely ignore the rest of the code in this cell

os.chdir("/home/sagemaker-user/s4casting/")  # change working directory, else the functions could misbehave

# simulate command-line arguments
sys.argv = [
    "scripts/train.py",  # training script
    "configs/cuda-tiny.toml",  # config
]

config = get_configuration()  # all the configuration from cuda-tiny.toml

machine = fc.provide_machine(config.machine, rng_base_seed=config.run.seed)
torch.manual_seed(machine.local_seed)
random.seed(machine.local_seed + 0x1FF)
np.random.seed(machine.local_seed + 0x2FF)

model_container = fc.provide_model_container(config.model, config.io, machine)
optimizer = fc.provide_optimizer(config.optimizer, model_container.raw_model.parameters())
scheduler = fc.provide_scheduler(config.scheduler, optimizer)
trainer = fc.provide_trainer(config=config.training, io=config.io, optimizer=config.optimizer, machine=machine)

batcher = fc.provide_batcher(
    config.io,
    config.model,
    config.training,
    config.validation,
    config.benchmarking,
    config.run,
    machine,
    trainer.hooks,
)

context = Context(
    configuration=config,
    model_container=model_container,
    optimizer=optimizer,
    scheduler=scheduler,
    machine=machine,
    trainer=trainer,
    batcher=batcher,
)

In the image below, you see how this model could be illustrated in a children's story book.  

You'll also see the structure of the input/outputs + model components. 

The input is shaped B * L * F with B = batch size, L = sequence length (number of time steps), F = number of features. 

- B - Batch size<br>
  Taken from ```config.training.batch_size```

- L - Sequence length<br>
  Computed as<br>

   ```(config.model.input_width_days + config.model.predict_width_days) * 60 minutes * 24 hours / config.model.input_sample_interval_minutes```
- F - number of features
  Derived from ```config.io.features```.
  In this tutorial (without weather features), it'll remain ```1```. When weather features are enabled, the number of selected weather is added.

In [ ]:
ipy_display(Image(filename="./notebooks/images/80sgogo.png"))

The entire model architecture can be broken down into four components -  

0. Input (context + prediction) - We load the time series as ```B * L * F ``` (batch, time, features). 

1. Patch encoder - We slice the time series into non-overlapping patches of length ```P``` (```config.model.patch_encoder.patch_size```). Each patch of ```P * F``` is flattened and mapped to a 256-dimensional vector (```config.model.latent_dim```). Patching compresses the length from ```L``` to ```L/P``` which makes learning cheaper and less sensitive to high-frequency noise (at the cost of lower time resolution). 

2. Backbone - S6 is a state-space sequence model. It reads the patch tokens in order while carrying a hidden state forward - the hidden state allows the model to model the sequence effectively. There are many [good explanations for S6/S4](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-mamba-and-state) posted online already, so please read [those](https://srush.github.io/annotated-s4/) or the [original paper](https://arxiv.org/abs/2312.00752). Crucially, unlike transformers, increasing the sequence length in S6 will only increase the compute by $O(L)$ instead of $O(L^2)$. In this tutorial, we use S6 as the backbone but you can also choose a GRU or transformer as the backbone. 

3. Patch decoder - It reverses the operations performed by the patch encoder by concatenating the current patch embedding with the next one, and then "unpatching" and mapping back to the original time step. The concatenation of two patches before unpatching helps smooth out the edges and prevent sharp jumps between patches.  

4. Output head - GMM, gaussian mixture model. Instead of a point prediction, the model outputs GMM parameters per time step. This means that we get probabilistic distributions for each time step. We train by maximizing the probability of the observed target window (i.e. minimizing its negative log-likelihood). 

In [ ]:
# You can examine the model here - you'll see that we're using 2 * S6 blocks in this tutorial
model_container.model

In [ ]:
# How big is this model?
# Only about 2 million parameters
total, trainable = count_parameters(model_container.model)
dtype = getattr(config.model, "internal_dtype", torch.float32)
print("=== Run Summary ===")
print(f"Model: {type(model_container.model).__name__}")
print(f"Parameters: total={total:,}, trainable={trainable:,} (~{mb(trainable):.1f} MB)")

## Build dataset
To train the model, we'll also need to build a dataset with our time series. The function ```PredictionTaskDataset``` does that by taking the forecasting task parameters -
- forecast dimension (within F, usually the first dimension (0) based on the order of config.io.features and in this tutorial it's the first and only feature)
- number of prediction steps
  
and providing the input (X), ground truth (Y) together with their respective masks (Xm, Ym).

In [ ]:
# How big is our dataset?
# This depends on how you're "slicing" it to predict_width_days
N = len(batcher.train)
print(
    f"The training dataset has {N} samples."
)  # note that this N will differ based on how your "slice" the dataset based on context/prediction time steps

X, Xm, Y, Ym = next(
    iter(
        DataLoader(
            batcher.train,
            batch_size=config.training.batch_size,
            num_workers=0,
            shuffle=True,
            pin_memory=False,
            persistent_workers=False,
        )
    )
)

X and Y have the same ```B * L * F``` shape, and so do the masks output by ```PredictionTaskDataset```.  

The target/prediction values in X are masked; in contrast, the input/context values in Y are masked. English can be confusing when describing this so look at the code and plot below.

In [ ]:
print(f"X is {X.shape}")  # B, L, F
print(f"Xm is {Xm.shape}")  # Input mask - 1 for input, 0 for prediction/target
print(f"Y is {Y.shape}")  # B, L, F
print(f"Ym is {Ym.shape}")  # Output mask - 0 for input values, 1 for prediction/target

# Count where masks are 1
input_length = (Xm[0] == 1).sum().item()
pred_length = (Ym[0] == 1).sum().item()

fig, ax = plt.subplots(figsize=(10, 2))
ax.barh(0, input_length, label=f"Context/Input: {input_length} steps")
ax.barh(0, pred_length, left=input_length, label=f"Prediction/target: {pred_length} steps")
ax.set_xlim(0, Xm.shape[1])
ax.set_yticks([])
ax.legend()
ax.set_xlabel("Time Steps")
ax.set_title("Context vs Prediction Window Split");
# note: sometimes you might get unlucky/lucky and get a sample with fewer context/prediction steps due to missing values
# if that's the case, you'll see the missing values in the plot below; during training, these missing values are masked

In [ ]:
# plotting the model predictions
# forecasting parameters
input_width_samples = (config.model.input_width_days * 24 * 60) // config.model.input_sample_interval_minutes
predict_width_samples = (config.model.predict_width_days * 24 * 60) // config.model.input_sample_interval_minutes

# how many days in both input & predict
days_per_sample = config.model.input_width_days + config.model.predict_width_days
samples_per_day = 96  # 24 hours * 4 samples/hour
pred_time = np.arange(input_width_samples, input_width_samples + predict_width_samples)  # time axis for predictions
pred_slice = slice(-predict_width_samples, None)

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-predict_width_samples].squeeze(-1), label="Example context")
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Example target window")

for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")
plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

## Example model prediction

In [ ]:
X, Xm, Y, Ym = next(
    iter(
        DataLoader(
            batcher.train,
            batch_size=config.training.batch_size,
            num_workers=0,
            shuffle=True,
            pin_memory=False,
            persistent_workers=False,
        )
    )
)
X = X.to(context.machine.torch_device).float()
Y = Y.to(context.machine.torch_device).float()
Xm = Xm.to(context.machine.torch_device).float()
Ym = Ym.to(context.machine.torch_device).float()
with torch.inference_mode():
    _prediction, loss = context.model_container.model(X, Xm, Y, Ym)

As this model is probablistic, instead of predicting a point value, the model predicts the value in different quantiles.  

If the output head is a Gaussian Mixture Model (GMM) as stated in the default configurations, the prediction will be shaped ```B * L * 1 * 4 gaussians * 3 parameters (mu, sigma, pi)```. These GMM parameters are then converted into quantile values during evaluation. 

If the output head is a quantile head, the prediction will be shaped ```B * L * 1 * number of quantiles```. You can access the quantiles in the quantile head via ```config.model.output_head.quantile_values``` - certainly, you could also specify desired quantiles in the config file.

>**Note**: In this tutorial, we are using a quantile output head.

The quantiles which the model will be evaluated on is determined in ```config.benchmarking.eval_quantiles``` - it seems redundant but we added this to the benchmark explicitly so that model performance is comparable. 

In short, you can specify more/less quantiles than what's stated but for benchmarking purposes, all your specified quantiles for the output head should overlap with the benchmarking.eval_quantiles.

In [ ]:
print(_prediction.shape)  # to extract only the prediction, make sure to slice L

print(f"The default quantiles are {config.model.output_head.quantile_values}")

print(f"The default evaluation quantiles are {config.benchmarking.eval_quantiles}")

In [ ]:
# this convert GMM parameters to quantiles
if context.configuration.model.output_head.arch == "gmm":
    logpi, sigma, mu = (x for x in _prediction.squeeze(2).unbind(dim=-1))
    logpi = torch.nan_to_num(logpi)
    sigma = torch.nan_to_num(sigma)
    mu = torch.nan_to_num(mu)
    qs = (
        gmm_to_quantiles(
            torch.exp(logpi),
            sigma,
            mu,
            config.benchmarking.eval_quantiles,
        )
        .detach()
        .cpu()
        .numpy()
    )
else:
    qs = _prediction[:, :, 0, :].detach().cpu().numpy()

In [ ]:
# plotting the model predictions
# forecasting parameters
input_width_samples = (config.model.input_width_days * 24 * 60) // config.model.input_sample_interval_minutes
predict_width_samples = (config.model.predict_width_days * 24 * 60) // config.model.input_sample_interval_minutes

# how many days in both input & predict
days_per_sample = config.model.input_width_days + config.model.predict_width_days
samples_per_day = 96  # 24 hours * 4 samples/hour
pred_time = np.arange(input_width_samples, input_width_samples + predict_width_samples)  # time axis for predictions
pred_slice = slice(-predict_width_samples, None)
median_idx = 6  # hardcoded :( - if the config changes, this will too

# move tensor to cpu
X = X.cpu().numpy()
Y = Y.cpu().numpy()

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-predict_width_samples].squeeze(-1), label="Example context")
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Example target window")
plt.plot(pred_time, qs[0, pred_slice, median_idx], label="Example (untrained/nonsensical) prediction")

# plot day separators
for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")

plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

The green lines are the models prediction of the 50th quantile. This is from our untrained model with randomly initialized weights, so of course these predictions don't make no sense. We're going to change that!

## Let's train the model

Running the cell below will take ~10 minutes. The comments in the cell describe the training steps. It's as easy as -
1. Get some data from batcher
2. Move data to GPU
3. Input data to model to get loss
4. Backpropagate loss through model
5. Repeat

After understanding the training loop, it's a good moment to take a break, touch grass and talk to other humans. As the model trains, a loss plot will show up and gets updated every 10 iteration. You can stare at the plot, or, switch tabs to hacker news. 

In our default configuration, a Mamba SSM with 6 layers and 1024 dimension, trained on 10 years of historical load measurements and historical weather data will take about 100k iterations to converge. 

In the plot below, you'll see that the loss drops quickly in the first 50 iterations then gradually levels off. 

In [ ]:
iteration = 0
maximum_train_steps = 1_000
losses = []
steps = []

context.optimizer.zero_grad(set_to_none=True)  # clear gradients from graph
plot_display_freq = 10
fig, ax = plt.subplots(figsize=(10, 5))

# basic training loop for you to see the model in action
for step in tqdm(
    range(maximum_train_steps)
):  # instead of config.training.maximum_steps to make it more easily configurable
    # loading data to device (cpu/gpu)
    X, Xm, Y, Ym = next(
        iter(
            DataLoader(
                batcher.train,
                batch_size=config.training.batch_size,
                num_workers=0,
                shuffle=True,
                pin_memory=False,
                persistent_workers=False,
            )
        )
    )
    X = X.to(context.machine.torch_device).float()
    Y = Y.to(context.machine.torch_device).float()
    Xm = Xm.to(context.machine.torch_device).float()
    Ym = Ym.to(context.machine.torch_device).float()

    # input data to model
    _prediction, loss = context.model_container.model(X, Xm, Y, Ym)
    loss.backward()

    # track model loss
    losses.append(loss.item())
    steps.append(iteration)

    # plot loss
    if step != 0 and step % plot_display_freq == 0:
        clear_output(wait=True)
        ax.clear()
        ax.plot(steps, losses, "k", linewidth=2, alpha=0.6, marker="o")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title(f"Training loss - step {iteration}: {loss.item():.3f}")
        ipy_display(fig)

    if trainer._optimizer.gradient_clipping:
        torch.nn.utils.clip_grad_norm_(context.model_container.model.parameters(), trainer._optimizer.gradient_clipping)

    context.optimizer.step()
    context.optimizer.zero_grad(set_to_none=True)
    iteration += 1  # noqa: SIM113

# How does the model do after 1000 steps?

You can run the next two cells multiple times to look at model predictions for different time series.

In [ ]:
# get the next batch of data
X_1k, Xm_1k, Y_1k, Ym_1k = next(
    iter(
        DataLoader(
            batcher.train,
            batch_size=config.training.batch_size,
            num_workers=0,
            shuffle=True,
            pin_memory=False,
            persistent_workers=False,
        )
    )
)

# move data to GPU
X_1k = X_1k.to(context.machine.torch_device).float()
Y_1k = Y_1k.to(context.machine.torch_device).float()
Xm_1k = Xm_1k.to(context.machine.torch_device).float()
Ym_1k = Ym_1k.to(context.machine.torch_device).float()

# input data to model
with torch.inference_mode():
    _prediction, loss = context.model_container.model(X_1k, Xm_1k, Y_1k, Ym_1k)

# convert predictions to quantiles
if context.configuration.model.output_head.arch == "gmm":
    logpi, sigma, mu = (x for x in _prediction.squeeze(2).unbind(dim=-1))
    logpi = torch.nan_to_num(logpi)
    sigma = torch.nan_to_num(sigma)
    mu = torch.nan_to_num(mu)
    qs = (
        gmm_to_quantiles(
            torch.exp(logpi),
            sigma,
            mu,
            config.model.output_head.quantile_values,
        )
        .detach()
        .cpu()
        .numpy()
    )
else:
    qs = _prediction[:, :, 0, :].detach().cpu().numpy()

In [ ]:
# plotting the model predictions
# forecasting parameters
input_width_samples = (config.model.input_width_days * 24 * 60) // config.model.input_sample_interval_minutes
predict_width_samples = (config.model.predict_width_days * 24 * 60) // config.model.input_sample_interval_minutes

# how many days in both input & predict
days_per_sample = config.model.input_width_days + config.model.predict_width_days
samples_per_day = 96  # 24 hours * 4 samples/hour
pred_time = np.arange(input_width_samples, input_width_samples + predict_width_samples)  # time axis for predictions
pred_slice = slice(-predict_width_samples, None)
median_idx = 6

# move tensor to cpu
X = X_1k.cpu().numpy()
Y = Y_1k.cpu().numpy()

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-predict_width_samples].squeeze(-1), label="Example context")
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Example target window")
plt.plot(pred_time, qs[0, pred_slice, median_idx], label="Example prediction at the 50th quantile")

# day separators
for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")

plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

The model that has been trained for 1000 iterations has learned the broad/rough measurement patterns. 

Especially for samples with strong periodicity, the model learns this within 1000 steps. 

Try running the two cells above a few more times to get a sense of what the model has learned. 

The observed performance is not bad for a model that has only been trained on ```config.training.batch_size * maximum_training_steps``` - if you didn't change the default config, that would be 128_000 samples which is ~2% of our dataset.

In the plots above, we are only visualizing the 50th quantiles, we can only observe the spread of probable values below.

There is a lot of code below, but it's mainly for making the plots pretty. There's no need to try to understand them. 

In [ ]:
# plotting the model predictions
# forecasting parameters
input_width_samples = (config.model.input_width_days * 24 * 60) // config.model.input_sample_interval_minutes
predict_width_samples = (config.model.predict_width_days * 24 * 60) // config.model.input_sample_interval_minutes

# how many days in both input & predict
days_per_sample = config.model.input_width_days + config.model.predict_width_days
samples_per_day = 96  # 24 hours * 4 samples/hour
pred_time = np.arange(input_width_samples, input_width_samples + predict_width_samples)  # time axis for predictions
pred_slice = slice(-predict_width_samples, None)

# get forecasting quantiles
quantiles = np.array(config.model.output_head.quantile_values, dtype=float)

# map value -> index in last dim of qs
q_to_idx = {float(q): i for i, q in enumerate(quantiles)}

# choose symmetric bands you want to draw (outer -> inner)
band_lowers = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40]
quantile_pairs = [
    (q, q_to_idx[q], 1 - q, q_to_idx[1 - q]) for q in band_lowers if q in q_to_idx and (1 - q) in q_to_idx
]

median_q = 0.5
median_idx = q_to_idx[median_q]

base_color = "steelblue"

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-predict_width_samples].squeeze(-1), label="Example context")
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Example target window")

# Plot quantile bands from widest to narrowest
for lower_q, lower_idx, upper_q, upper_idx in quantile_pairs:
    # Extract quantiles - shape is [batch, time, channels, quantiles]
    lower_bound = qs[0, pred_slice, lower_idx]
    upper_bound = qs[0, pred_slice, upper_idx]

    # Calculate alpha based on distance from median
    alpha = get_quantile_alpha(lower_q)

    # Create filled band
    plt.fill_between(
        pred_time,
        lower_bound,
        upper_bound,
        alpha=alpha,
        color=base_color,
        label=f"{int(lower_q * 100)}%-{int(upper_q * 100)}%",
        edgecolor=base_color,
        linewidth=0.5,
    )

plt.plot(pred_time, qs[0, pred_slice, median_idx], color="darkblue", linewidth=2, label="Median (50%)", zorder=10)

for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")
plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Ground truth", color="orange")
for lower_q, lower_idx, upper_q, upper_idx in quantile_pairs:
    # Extract quantiles - shape is [batch, time, channels, quantiles]
    lower_bound = qs[0, pred_slice, lower_idx]
    upper_bound = qs[0, pred_slice, upper_idx]

    # Calculate alpha based on distance from median
    alpha = get_quantile_alpha(lower_q)

    # Create filled band
    plt.fill_between(
        pred_time,
        lower_bound,
        upper_bound,
        alpha=alpha,
        color=base_color,
        label=f"{int(lower_q * 100)}%-{int(upper_q * 100)}%",
        edgecolor=base_color,
        linewidth=0.5,
    )

plt.plot(pred_time, qs[0, pred_slice, median_idx], color="darkblue", linewidth=2, label="Median (50%)", zorder=10)

plt.title("Close up on prediction")
plt.legend()

In the close up on prediction, we see that the median prediction tracks the ground truth reasonably well (for most of the samples at least, for certain signals the model just fails because it has not been trained long enough).

## What about a model that has been trained for 100_000 steps?

In the cell below, we will reinitialize the model and load a model checkpoint that we have trained for 100k steps.
> **Note** if you changed any modules from the defaults as specified in cuda-tiny.toml, the weights "will not fit" anymore

In [ ]:
# Build context from config
# simulate command-line arguments
sys.argv = [
    "scripts/train.py",  # training script
    "configs/cuda-tiny.toml",  # config
]

# get configuration
with pathlib.Path("configs/cuda-tiny.toml").open("r", encoding="utf-8") as f:
    config_data = tomlkit.load(f)

# here we are loading a model that has been trained for 100k
config_data["io"]["load_checkpoint"] = "notebooks/checkpoint/checkpoint_tiny_100000.pt"

config = Configuration(**config_data.unwrap())

machine = fc.provide_machine(config.machine, rng_base_seed=config.run.seed)
torch.manual_seed(machine.local_seed)
random.seed(machine.local_seed + 0x1FF)
np.random.seed(machine.local_seed + 0x2FF)

model_container = fc.provide_model_container(config.model, config.io, machine)
optimizer = fc.provide_optimizer(config.optimizer, model_container.raw_model.parameters())
scheduler = fc.provide_scheduler(config.scheduler, optimizer)
trainer = fc.provide_trainer(config=config.training, io=config.io, optimizer=config.optimizer, machine=machine)
checkpointer = fc.provide_checkpointer(config.io, trainer.hooks)
batcher = fc.provide_batcher(
    config.io,
    config.model,
    config.training,
    config.validation,
    config.benchmarking,
    config.run,
    machine,
    trainer.hooks,
)

context = Context(
    configuration=config,
    model_container=model_container,
    optimizer=optimizer,
    scheduler=scheduler,
    machine=machine,
    trainer=trainer,
    checkpointer=checkpointer,
    batcher=batcher,
)

checkpointer.load(context)  # loads a model which has been trained for 100k steps

Now let's load in the same sample that we plotted above to see how much our model has improved.

In [ ]:
# move data back to GPU
X_1k = X_1k.to(context.machine.torch_device).float()
Y_1k = Y_1k.to(context.machine.torch_device).float()
Xm_1k = Xm_1k.to(context.machine.torch_device).float()
Ym_1k = Ym_1k.to(context.machine.torch_device).float()

# input data to model
with torch.inference_mode():
    _prediction, loss = context.model_container.model(X_1k, Xm_1k, Y_1k, Ym_1k)

# convert predictions to quantiles
if context.configuration.model.output_head.arch == "gmm":
    logpi, sigma, mu = (x for x in _prediction.squeeze(2).unbind(dim=-1))
    logpi = torch.nan_to_num(logpi)
    sigma = torch.nan_to_num(sigma)
    mu = torch.nan_to_num(mu)
    qs = (
        gmm_to_quantiles(
            torch.exp(logpi),
            sigma,
            mu,
            config.model.output_head.quantile_values,
        )
        .detach()
        .cpu()
        .numpy()
    )
else:
    qs = _prediction[:, :, 0, :].detach().cpu().numpy()

In [ ]:
# plotting the model predictions
# forecasting parameters
input_width_samples = (config.model.input_width_days * 24 * 60) // config.model.input_sample_interval_minutes
predict_width_samples = (config.model.predict_width_days * 24 * 60) // config.model.input_sample_interval_minutes

# how many days in both input & predict
days_per_sample = config.model.input_width_days + config.model.predict_width_days
samples_per_day = 96  # 24 hours * 4 samples/hour
pred_time = np.arange(input_width_samples, input_width_samples + predict_width_samples)  # time axis for predictions
pred_slice = slice(-predict_width_samples, None)

# get forecasting quantiles
quantiles = np.array(config.model.output_head.quantile_values, dtype=float)

# map value -> index in last dim of qs
q_to_idx = {float(q): i for i, q in enumerate(quantiles)}

# choose symmetric bands you want to draw (outer -> inner)
band_lowers = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40]
quantile_pairs = [
    (q, q_to_idx[q], 1 - q, q_to_idx[1 - q]) for q in band_lowers if q in q_to_idx and (1 - q) in q_to_idx
]

median_q = 0.5
median_idx = q_to_idx[median_q]

# move tensor to cpu
X = X_1k.cpu().numpy()
Y = Y_1k.cpu().numpy()

base_color = "steelblue"

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-predict_width_samples].squeeze(-1), label="Example context")
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Example target window")

# Plot quantile bands from widest to narrowest
for lower_q, lower_idx, upper_q, upper_idx in quantile_pairs:
    # Extract quantiles - shape is [batch, time, channels, quantiles]
    lower_bound = qs[0, pred_slice, lower_idx]
    upper_bound = qs[0, pred_slice, upper_idx]

    # Calculate alpha based on distance from median
    alpha = get_quantile_alpha(lower_q)

    # Create filled band
    plt.fill_between(
        pred_time,
        lower_bound,
        upper_bound,
        alpha=alpha,
        color=base_color,
        label=f"{int(lower_q * 100)}%-{int(upper_q * 100)}%",
        edgecolor=base_color,
        linewidth=0.5,
    )

plt.plot(pred_time, qs[0, pred_slice, median_idx], color="darkblue", linewidth=2, label="Median (50%)", zorder=10)

for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")
plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

It depends on which sample you're plotting, for most signals, you should see improvements after training for longer. However, for solar/wind related signals, the improvements will be minimal because the model has not been trained to associate the load with weather data.

> **Try it out**: Try running the following two cells multiple times to get a feel of the model's behavior.

In [ ]:
# get some data
X, Xm, Y, Ym = next(
    iter(
        DataLoader(
            batcher.train,
            batch_size=config.training.batch_size,
            num_workers=0,
            shuffle=True,
            pin_memory=False,
            persistent_workers=False,
        )
    )
)

# move data to GPU
X = X.to(context.machine.torch_device).float()
Y = Y.to(context.machine.torch_device).float()
Xm = Xm.to(context.machine.torch_device).float()
Ym = Ym.to(context.machine.torch_device).float()

# input data to model
with torch.inference_mode():
    _prediction, loss = context.model_container.model(X, Xm, Y, Ym)

# convert predictions to quantiles
if context.configuration.model.output_head.arch == "gmm":
    logpi, sigma, mu = (x for x in _prediction.squeeze(2).unbind(dim=-1))
    logpi = torch.nan_to_num(logpi)
    sigma = torch.nan_to_num(sigma)
    mu = torch.nan_to_num(mu)
    qs = (
        gmm_to_quantiles(
            torch.exp(logpi),
            sigma,
            mu,
            config.model.output_head.quantile_values,
        )
        .detach()
        .cpu()
        .numpy()
    )
else:
    qs = _prediction[:, :, 0, :].detach().cpu().numpy()

In [ ]:
# plotting the model predictions
# forecasting parameters
input_width_samples = (config.model.input_width_days * 24 * 60) // config.model.input_sample_interval_minutes
predict_width_samples = (config.model.predict_width_days * 24 * 60) // config.model.input_sample_interval_minutes

# how many days in both input & predict
days_per_sample = config.model.input_width_days + config.model.predict_width_days
samples_per_day = 96  # 24 hours * 4 samples/hour
pred_time = np.arange(input_width_samples, input_width_samples + predict_width_samples)  # time axis for predictions
pred_slice = slice(-predict_width_samples, None)

# get forecasting quantiles
quantiles = np.array(config.model.output_head.quantile_values, dtype=float)

# map value -> index in last dim of qs
q_to_idx = {float(q): i for i, q in enumerate(quantiles)}

# choose symmetric bands you want to draw (outer -> inner)
band_lowers = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40]
quantile_pairs = [
    (q, q_to_idx[q], 1 - q, q_to_idx[1 - q]) for q in band_lowers if q in q_to_idx and (1 - q) in q_to_idx
]

median_q = 0.5
median_idx = q_to_idx[median_q]

# move tensor to cpu
X = X.cpu().numpy()
Y = Y.cpu().numpy()

base_color = "steelblue"

plt.figure(figsize=(20, 6))
plt.plot(np.arange(input_width_samples), X[0, :-predict_width_samples].squeeze(-1), label="Example context")
plt.plot(pred_time, Y[0, pred_slice].squeeze(-1), label="Example target window")

# Plot quantile bands from widest to narrowest
for lower_q, lower_idx, upper_q, upper_idx in quantile_pairs:
    # Extract quantiles - shape is [batch, time, channels, quantiles]
    lower_bound = qs[0, pred_slice, lower_idx]
    upper_bound = qs[0, pred_slice, upper_idx]

    # Calculate alpha based on distance from median
    alpha = get_quantile_alpha(lower_q)

    # Create filled band
    plt.fill_between(
        pred_time,
        lower_bound,
        upper_bound,
        alpha=alpha,
        color=base_color,
        label=f"{int(lower_q * 100)}%-{int(upper_q * 100)}%",
        edgecolor=base_color,
        linewidth=0.5,
    )

plt.plot(pred_time, qs[0, pred_slice, median_idx], color="darkblue", linewidth=2, label="Median (50%)", zorder=10)

for day in range(0, days_per_sample):
    x_pos = day * samples_per_day
    plt.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.3)
    plt.text(x_pos + samples_per_day / 2, plt.ylim()[0], f"Day {day + 1}", ha="center", va="bottom")
plt.title("Training data sample")
plt.xlabel("Time")
plt.ylabel("Load")
plt.legend()
plt.show()

You should see that the model is better able to follow the higher frequency changes (the 'spikiness'/volatility of targets) than with the model that's only trained for 1000 steps. At 10k steps, the model better captures the timing, shape and amplitude of peaks. Though for signals which don't have a clear daily cycles, the model performs less well. This is something we can improve by making the context bigger (e.g. instead of 7 days, perhaps 30 days) - effectively allowing the model to see further back. 

If you run the above two cells a few more times, you'll also see that for signals with wind/solar generation, the model also performs poorly because it was not trained with weather data. This is also something we can improve, by training the model with weather data. 

So now it's your turn to do this, check out other notebooks on how to change the training parameters, training data and model modules. 

## Conclusion: s4casting

In this notebook, we explored the s4casting repository to train a state space model on short-term load forecasting. With a small model of 2 layers and 256 dimensions and training for 1000 steps, we can already get reasonable predictions. 

During model training, we observed that the model learns quickly, especially for signals with strong periodicity. The model outputs its predictions in quantiles, enabling probabalistic predictions. The probabalistic values allow for calibrated risk-taking on congestion management. 

Now it's your turn to change things and break records on our benchmark. Have fun!